In [ ]:
# 3 clinical validation 
import os 
import glob
from tqdm import tqdm
import nibabel as nib
import numpy as np
import neuroimage_analysis as na
import matplotlib.pyplot as plt 
import pandas as pd
from scipy.stats import pearsonr
import seaborn as sns


## Analysis 3: Clinical Prediction — sLNM vs. Gradient 1

To assess the clinical utility of sLNM, we compare its predictive value against gradient 1 — the first principal component of the GSP1000 healthy reference connectome — which is entirely independent of patient data. We use the same Broca's aphasia (WAB-AQ) and depression-TMS (BDI change) datasets from Analysis 1.

**Leave-one-out prediction**
For each left-out subject, the sLNM map is computed from all remaining subjects. The prediction is defined as the spatial correlation between the held-out subject's FC map and the group sLNM map. LOO cross-validation was chosen to match the standard approach used in prior sLNM validation studies.

**Gradient 1 comparison**
We repeat the same procedure using gradient 1 as a fixed reference map in place of the sLNM map. Because gradient 1 is derived purely from healthy controls, any predictive performance reflects general connectome structure rather than disease-specific circuitry.

**Outcome**
Predictive performance (spatial r with symptom severity) is compared between sLNM and gradient 1 across both datasets.

In [ ]:
dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
outdir = os.path.join(dir, "results")

brain_template = nib.load(os.path.join(dir, "data/templates/Taylor_NHB_MNI152_T1_2mm_brain_mask_dil.nii.gz"))
brain_mask = brain_template.get_fdata() > 0



def get_slnm(fc_paths, behavior, visualize=True, outdir=None, filename=None):
    """
    Compute sLNM map by correlating FC maps with a behavior vector.
    If visualize=True, saves NIfTI and CIFTI surface files to outdir.
    """
    fc_array = np.hstack([nib.load(f).get_fdata()[brain_mask].reshape(-1, 1) for f in fc_paths]).T
    behavior = np.array(behavior).reshape(-1, 1)
    lnm_map = na.voxel_outcome_correlation(fc_array, behavior)

    if visualize:
        brain_3d = np.zeros_like(brain_template.get_fdata())
        brain_3d[brain_mask] = lnm_map
        lnm_img = nib.Nifti1Image(brain_3d, affine=brain_template.affine, header=brain_template.header)
        out_path = os.path.join(outdir, f'{filename}.nii.gz')
        nib.save(lnm_img, out_path)
        print(f'Saved: {out_path}')
        na.nifti_to_cifti(nii_path=out_path, outdir=outdir)

    return lnm_map


def loo_prediction(fc_files, behavior):
    """
    Leave-one-out prediction using sLNM.

    For each subject, computes an sLNM map from all other subjects, then
    measures the spatial similarity (similarity_r) between that map and the left-out
    subject's FC map. Returns an array of [behavior, similarity_r].
    """
    similarity_r = []

    for i in tqdm(range(len(fc_files)), desc='LOO analysis'):
        # Leave one subject out
        train_files    = fc_files[:i] + fc_files[i+1:]
        train_behavior = behavior[:i] + behavior[i+1:]

        # Compute sLNM from training subjects
        loo_network = get_slnm(fc_paths=train_files, behavior=train_behavior, visualize=False)

        # Correlate left-out subject's FC map with the LOO sLNM map
        r = pearsonr(na.nifti_getdata(fc_files[i]), loo_network).statistic
        similarity_r.append(r)

    return np.column_stack([behavior, similarity_r])


def pc1_validation(fc_surf_files, behavior, pc1_map=None):
    """
    Validate behavior prediction using PC1 (Gradient 1) of the GSP connectome.

    PC1 is in fslr-32k surface space, so subject FC files should be resampled in fslr-32k before analysis. 
    Returns a (n_subjects x 2) array of [behavior, similarity_r].
    """
    if pc1_map is None:
        pc1_path = os.path.join(dir, 'data/templates/gradient_1.dscalar.nii')
        pc1_map = nib.load(pc1_path).get_fdata()

    similarity_r = []
    for file in tqdm(fc_surf_files, desc='Calculating similarity to PC1'):
        surf_data = nib.load(file).get_fdata()
        r = na.pearson_rows(surf_data, pc1_map)
        similarity_r.append(r)

    return np.column_stack([behavior, similarity_r])




In [ ]:
# ── Load TMS dataset ──────────────────────────────────────────────────────────
tms_dataset = os.path.join(dir, "data/tms_dataset")

tms_fc_files = sorted(glob.glob(os.path.join(tms_dataset, '*AvgR.nii.gz')))
print(f'Found {len(tms_fc_files)} TMS FC files')

df_tms = pd.read_csv(os.path.join(tms_dataset, 'tms_subjects.csv'))
df_tms = df_tms.sort_values('Patient').reset_index(drop=True)
print(df_tms[['Patient', 'BDI_change_percent']])

bdi_changed = df_tms['BDI_change_percent'].tolist()

# ── Load ARC dataset (Broca subjects only) ────────────────────────────────────
arc_dataset = os.path.join(dir, "data/arc_dataset")

df_arc = pd.read_csv(os.path.join(arc_dataset, 'participants.tsv'), sep='\t')
df_arc = df_arc.sort_values('participant_id').reset_index(drop=True)

arc_fc_files = sorted(glob.glob(os.path.join(arc_dataset, '*sub*AvgR.nii.gz')))
arc_sub_fcid = [os.path.basename(f).split('_')[0].replace('w', '') for f in arc_fc_files]
print(f'Found {len(arc_fc_files)} subjects with FC files')

df_arc = df_arc[df_arc['participant_id'].isin(arc_sub_fcid)]
df_arc_broca = df_arc[df_arc['wab_type'] == 'Broca'].reset_index(drop=True)
print(f'Found {len(df_arc_broca)} Broca aphasia subjects')
print(df_arc_broca[['participant_id', 'wab_aq']])

sub_broca = df_arc_broca['participant_id'].tolist()
broca_fc_files = [f for f in arc_fc_files if os.path.basename(f).split('_')[0].replace('w', '') in sub_broca]
print(f'Found {len(broca_fc_files)} Broca FC files')

wab_aq = df_arc_broca['wab_aq'].tolist()

# ── Run LOO predictions ───────────────────────────────────────────────────────
tms_loo    = loo_prediction(tms_fc_files, bdi_changed)
broca_loo  = loo_prediction(broca_fc_files, wab_aq)

# ── Run PC1 validation (surface FC files) ────────────────────────────────────
tms_surf_files   = sorted(glob.glob(os.path.join(tms_dataset, '*AvgR*dscalar.nii')))
broca_surf_files = sorted(glob.glob(os.path.join(arc_dataset, '*sub*AvgR*dscalar.nii')))
print(f'Found {len(broca_surf_files)} Broca surface FC files')

tms_pc1   = pc1_validation(tms_surf_files, bdi_changed)
broca_pc1 = pc1_validation(broca_surf_files, wab_aq)



In [ ]:
plt.rcParams['font.size'] = 15
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharey='row')

# Row 1: TMS | Row 2: Broca
# Col 1: sLNM | Col 2: Gradient 1

datasets = [
    (tms_loo,   tms_pc1,   '% Reduction in BDI', 'Depression-TMS Network', 'Gradient 1'),
    (broca_loo, broca_pc1, 'WAB-AQ',              'Broca Network',          'Gradient 1'),
]

colors = ['dodgerblue', 'red']

for row, (slnm_result, pc1_result, xlabel, slnm_title, pc1_title) in enumerate(datasets):
    for col, (result, color, title) in enumerate(zip(
        [slnm_result, pc1_result],
        colors,
        [slnm_title, pc1_title]
    )):
        ax = axes[row, col]
        x, y = result[:, 0], result[:, 1]
        r, _ = pearsonr(x, y)
        if r < 0:
            y, r = -y, -r

        sns.regplot(x=x, y=y, ci=95, line_kws={'color': color}, ax=ax)
        for collection in ax.collections:
            collection.set_alpha(0.3)
        ax.plot(x, np.poly1d(np.polyfit(x, y, 1))(x), color='black', linewidth=2, zorder=2)
        ax.scatter(x, y, s=70, c=color, zorder=3)
        ax.set_title(f'{title}\n(r = {r:.2f})')
        ax.set_xlabel(xlabel)
        ax.locator_params(axis='y', nbins=5)

    axes[row, 0].set_ylabel('Similarity to Map')

plt.tight_layout()
plt.savefig(os.path.join(outdir, 'prediction_comparison.svg'), format='svg', bbox_inches='tight')
plt.show()